# Stage C1 — Physics Constraints

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Sec. 3.3 (Table 7, Eq. 3.7 and 3.9–3.15).

**Physics starts here.** B1/B2 were deliberately physics-free; this is
where the six constraint penalties get implemented and — just as
important — unit-tested against functions with a **known** derivative
sign, so a sign bug gets caught in seconds here instead of after a
multi-hour training run in C4.

**Input:** `data/masters_data.xlsx` (for the density/correlation
reference used by Section 8's adaptive weighting only — the
constraints themselves are evaluated on collocation points, not
training data).
**Output:** six tested penalty functions, 1000 LHS collocation points,
and the region-specific weighting machinery (Eq. 3.15) — all consumed
by **C2** to assemble the composite loss.

**Framework note:** the constraints differentiate a model's output
with respect to its *input*, which needs `tf.GradientTape` — not
something `polars`/`plotly` can do. Two of the six need a **second**
derivative (nested tapes) and two need **two separate** gradients from
the same tape (`persistent=True`) — both flagged explicitly below,
since they're the two easiest places to introduce a silent bug.


## Setup

In [ ]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from scipy.stats import qmc, gaussian_kde
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42
RAW_PATH


## 1. Load & normalize (same logic as A3/B1/B2)

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)

# index constants, used throughout instead of magic numbers
SOI_IDX, LAMBDA_IDX, SUBRATE_IDX, PRAIL_IDX = [INPUT_COLS.index(c) for c in INPUT_COLS]
HC_IDX, NOX_IDX, CO2_IDX, PM_IDX, ETA_IDX = [OUTPUT_COLS.index(c) for c in OUTPUT_COLS]
EMISSION_IDXS = [HC_IDX, NOX_IDX, CO2_IDX, PM_IDX]  # eta excluded -- bounded by sigmoid instead, Sec 3.3.3.1

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]

medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2
rng_np = np.random.default_rng(SEED)
jitter = rng_np.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))

train_df = df.filter(pl.col("split") == "train")
train_min = {c: train_df[c].min() for c in ALL_COLS}
train_max = {c: train_df[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])
X_train = df.filter(pl.col("split") == "train").select([f"{c}_norm" for c in INPUT_COLS]).to_numpy()
print("train inputs for the density/correlation reference:", X_train.shape)


## 2. Latin Hypercube collocation sampling

1000 points across the normalized 4D input domain $[0,1]^4$ — this is
where the physics constraints get evaluated (Sec. 3.2.2.2), everywhere
in the domain, not only at the 40 experimental points.

In [ ]:
N_COLLOC = 1000
sampler = qmc.LatinHypercube(d=N_IN, seed=SEED)
X_colloc = sampler.random(n=N_COLLOC).astype(np.float32)
print(X_colloc.shape, X_colloc.min(), X_colloc.max())


**Coverage check** — pairwise projections of the 1000 collocation points against the 28 training points:

In [ ]:
fig = make_subplots(rows=2, cols=3, subplot_titles=[
    f"{a} vs {b}" for i, a in enumerate(INPUT_COLS) for b in INPUT_COLS[i+1:]
])
pairs = [(i, j) for i in range(N_IN) for j in range(i + 1, N_IN)]
for k, (i, j) in enumerate(pairs):
    r, c = divmod(k, 3)
    fig.add_trace(go.Scatter(x=X_colloc[:, i], y=X_colloc[:, j], mode="markers",
                              marker=dict(color="#B7C9DA", size=4), name="collocation",
                              showlegend=(k == 0)), row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=X_train[:, i], y=X_train[:, j], mode="markers",
                              marker=dict(color="#993C1D", size=8), name="train points",
                              showlegend=(k == 0)), row=r + 1, col=c + 1)
fig.update_layout(height=550, width=950, title_text="LHS collocation coverage vs. training data (normalized)")
fig.show()


## 3. Synthetic test harness

Builds a fake 5-output "model" from simple formulas, so each
constraint can be checked against a case with a **known** correct sign
before it ever sees a real network. `formulas` maps an output index to
a function of `x`; any output not listed is filled with zeros (its
gradient is irrelevant to the test at hand).

In [ ]:
def synthetic_output(x, formulas):
    cols = []
    for i in range(N_OUT):
        cols.append(formulas[i](x) if i in formulas else tf.zeros(tf.shape(x)[0]))
    return tf.stack(cols, axis=1)

def run_case(constraint_fn, predict_fn, x, label):
    penalty = float(constraint_fn(predict_fn, x))
    print(f"  {label:10s} penalty = {penalty:.6f}")
    return penalty

X_test_pts = tf.constant(np.random.default_rng(0).uniform(0, 1, (100, N_IN)), dtype=tf.float32)


## 4. Monotonic constraints — NOx vs. SOI (Eq. 3.9) and PM vs. λ (Eq. 3.10)

$$
\mathcal{L}_{\text{NOx-SOI}} = \frac{1}{N_c}\sum_{i=1}^{N_c} \max\!\left(0,\ \frac{\partial \text{NOx}}{\partial \text{SOI}}\Big|_i\right)^{2}
\qquad
\mathcal{L}_{\text{PM-}\lambda} = \frac{1}{N_c}\sum_{i=1}^{N_c} \max\!\left(0,\ \frac{\partial \text{PM}}{\partial \lambda}\Big|_i\right)^{2}
$$

Both expect a **decreasing** relationship, so positive derivatives get
penalized, negative ones don't — same formula, different (output,
input) pair, implemented once and reused.

In [ ]:
def monotonic_decreasing_constraint(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        target = y[:, out_idx]
    grad = tape.gradient(target, x_t)
    d = grad[:, in_idx]
    return tf.reduce_mean(tf.square(tf.maximum(0.0, d)))

def nox_soi_constraint(predict_fn, x):
    return monotonic_decreasing_constraint(predict_fn, x, NOX_IDX, SOI_IDX)

def pm_lambda_constraint(predict_fn, x):
    return monotonic_decreasing_constraint(predict_fn, x, PM_IDX, LAMBDA_IDX)

print("NOx-SOI (Eq. 3.9):")
compliant = lambda x: synthetic_output(x, {NOX_IDX: lambda x: -x[:, SOI_IDX]})   # NOx decreases with SOI
violating = lambda x: synthetic_output(x, {NOX_IDX: lambda x: x[:, SOI_IDX]})    # NOx increases with SOI
run_case(nox_soi_constraint, compliant, X_test_pts, "compliant")
run_case(nox_soi_constraint, violating, X_test_pts, "violating")

print("\nPM-lambda (Eq. 3.10):")
compliant = lambda x: synthetic_output(x, {PM_IDX: lambda x: -x[:, LAMBDA_IDX]})
violating = lambda x: synthetic_output(x, {PM_IDX: lambda x: x[:, LAMBDA_IDX]})
run_case(pm_lambda_constraint, compliant, X_test_pts, "compliant")
run_case(pm_lambda_constraint, violating, X_test_pts, "violating")


## 5. Shape constraint — HC vs. λ convexity (Eq. 3.11)

$$
\mathcal{L}_{\text{HC-}\lambda} = \frac{1}{N_c}\sum_{i=1}^{N_c} \max\!\left(0,\ -\frac{\partial^2 \text{HC}}{\partial \lambda^2}\Big|_i\right)^{2}
$$

**Second** derivative — needs a tape *inside* a tape: the inner tape
gets $\partial \text{HC}/\partial\lambda$, the outer tape differentiates
that result again.

In [ ]:
def convexity_constraint(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            y = predict_fn(x_t)
            target = y[:, out_idx]
        grad1 = tape1.gradient(target, x_t)
        d_first = grad1[:, in_idx]
    grad2 = tape2.gradient(d_first, x_t)
    d_second = grad2[:, in_idx]
    return tf.reduce_mean(tf.square(tf.maximum(0.0, -d_second)))

def hc_lambda_constraint(predict_fn, x):
    return convexity_constraint(predict_fn, x, HC_IDX, LAMBDA_IDX)

print("HC-lambda (Eq. 3.11):")
compliant = lambda x: synthetic_output(x, {HC_IDX: lambda x: (x[:, LAMBDA_IDX] - 0.5) ** 2})   # convex, U-shaped
violating = lambda x: synthetic_output(x, {HC_IDX: lambda x: -(x[:, LAMBDA_IDX] - 0.5) ** 2})  # concave
run_case(hc_lambda_constraint, compliant, X_test_pts, "compliant")
run_case(hc_lambda_constraint, violating, X_test_pts, "violating")


## 6. Trade-off constraints — NOx–PM (Eq. 3.12) and η–NOx (Eq. 3.13)

$$
\mathcal{L}_{\text{NOx-PM}} = \frac{1}{N_c}\sum_{i=1}^{N_c} \max\!\left(0,\ \frac{\partial \text{NOx}}{\partial \text{SOI}}\Big|_i \cdot \frac{\partial \text{PM}}{\partial \text{SOI}}\Big|_i\right)^{2}
$$

Both derivatives are with respect to **SOI** specifically (not a
generic input) — penalizes them moving in the *same* direction. Needs
**two** separate `tape.gradient()` calls from the same tape, so the
tape must be `persistent=True` (and explicitly deleted after, standard
practice to free its resources). η–NOx (Eq. 3.13) reuses the exact
same function with a different output pair; the text calls it a
*softer* constraint, meaning a smaller base weight and region-specific
validity (Section 8), not a different formula.

In [ ]:
def tradeoff_constraint(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a = y[:, out_idx_a]
        b = y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return tf.reduce_mean(tf.square(tf.maximum(0.0, grad_a * grad_b)))

def nox_pm_tradeoff_constraint(predict_fn, x):
    return tradeoff_constraint(predict_fn, x, NOX_IDX, PM_IDX, SOI_IDX)

def eff_nox_tradeoff_constraint(predict_fn, x):
    return tradeoff_constraint(predict_fn, x, ETA_IDX, NOX_IDX, SOI_IDX)

print("NOx-PM (Eq. 3.12):")
compliant = lambda x: synthetic_output(x, {NOX_IDX: lambda x: x[:, SOI_IDX], PM_IDX: lambda x: -x[:, SOI_IDX]})
violating = lambda x: synthetic_output(x, {NOX_IDX: lambda x: x[:, SOI_IDX], PM_IDX: lambda x: x[:, SOI_IDX]})
run_case(nox_pm_tradeoff_constraint, compliant, X_test_pts, "compliant")
run_case(nox_pm_tradeoff_constraint, violating, X_test_pts, "violating")

print("\neta-NOx (Eq. 3.13):")
compliant = lambda x: synthetic_output(x, {ETA_IDX: lambda x: -x[:, SOI_IDX], NOX_IDX: lambda x: x[:, SOI_IDX]})
violating = lambda x: synthetic_output(x, {ETA_IDX: lambda x: x[:, SOI_IDX], NOX_IDX: lambda x: x[:, SOI_IDX]})
run_case(eff_nox_tradeoff_constraint, compliant, X_test_pts, "compliant")
run_case(eff_nox_tradeoff_constraint, violating, X_test_pts, "violating")


## 7. Non-negativity (Eq. 3.14)

$$
\mathcal{L}_{\text{nonneg}} = \frac{1}{N}\sum_{i=1}^{N}\sum_{k \in \{\text{HC,NOx,CO2,PM}\}} \max(0,\ -y^{\text{pred}}_{i,k})^{2}
$$

Only these four — **eta is excluded**: Sec. 3.3.3.1 bounds it to
$[0,1]$ architecturally via a sigmoid output activation instead of a
loss penalty, so it never needs this term. No gradient here, just the
predicted values themselves.

In [ ]:
def nonneg_constraint(predict_fn, x, out_idxs=EMISSION_IDXS):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    y = predict_fn(x_t)
    emissions = tf.gather(y, out_idxs, axis=1)
    return tf.reduce_mean(tf.square(tf.maximum(0.0, -emissions)))

print("Non-negativity (Eq. 3.14):")
compliant = lambda x: synthetic_output(x, {i: (lambda x: tf.ones(tf.shape(x)[0])) for i in EMISSION_IDXS})
violating = lambda x: synthetic_output(x, {i: (lambda x: -tf.ones(tf.shape(x)[0])) for i in EMISSION_IDXS})
run_case(nonneg_constraint, compliant, X_test_pts, "compliant")
run_case(nonneg_constraint, violating, X_test_pts, "violating")


## 8. Region-specific validity & adaptive weighting (Eq. 3.15)

$$
\lambda_j(x) = \lambda_j^{0} \cdot \frac{\rho(x)}{\rho_{\max} + \epsilon} \cdot v_j(x)
$$

Three pieces, each computed from the **training data only**:

- $\rho(x)$ — local data density (Gaussian KDE over the 28 training
  points), normalized by its own max, so the constraint relaxes far
  from any experimental evidence.
- $v_j(x)$ — validity function. The text specifies $v_j(x)=1$
  everywhere for the four high-confidence constraints (NOx-SOI, PM-λ,
  HC-λ, NOx-PM), and for η-NOx specifically, a value that tracks local
  correlation strength, approaching 1 where $|\rho_{\text{local}}|>0.5$
  and 0 where it's weak. **Implementation choice, not fully specified
  in the text:** "local" correlation is computed here as a
  Gaussian-kernel-weighted Pearson correlation around each query
  point — a defensible reading, but a reading, since Sec. 3.3.4.1
  doesn't give the exact estimator.
- $\lambda_j^0$ — base weight per constraint, set in **C2** alongside
  the other loss weights, not here.

In [ ]:
kde_train = gaussian_kde(X_train.T)

def density_ratio(x_query):
    rho = kde_train(x_query.T)
    return rho / (rho.max() + 1e-8) if rho.max() > 0 else rho

def local_correlation(x_query, x_ref, a_ref, b_ref, bandwidth=0.25):
    d = np.linalg.norm(x_ref[None, :, :] - x_query[:, None, :], axis=2)  # [n_query, n_ref]
    w = np.exp(-0.5 * (d / bandwidth) ** 2)
    w = w / (w.sum(axis=1, keepdims=True) + 1e-12)
    a_mean = (w * a_ref[None, :]).sum(axis=1)
    b_mean = (w * b_ref[None, :]).sum(axis=1)
    cov = (w * (a_ref[None, :] - a_mean[:, None]) * (b_ref[None, :] - b_mean[:, None])).sum(axis=1)
    var_a = (w * (a_ref[None, :] - a_mean[:, None]) ** 2).sum(axis=1)
    var_b = (w * (b_ref[None, :] - b_mean[:, None]) ** 2).sum(axis=1)
    return cov / np.sqrt(var_a * var_b + 1e-12)

eta_train = df.filter(pl.col("split") == "train")["eta_norm"].to_numpy()
nox_train = df.filter(pl.col("split") == "train")["NOx_norm"].to_numpy()

rho_colloc = density_ratio(X_colloc)
corr_colloc = local_correlation(X_colloc, X_train, eta_train, nox_train)
validity_eff_nox = np.clip((np.abs(corr_colloc) - 0.0) / 0.5, 0, 1)  # ->1 as |corr| exceeds 0.5

print(f"density ratio  : min={rho_colloc.min():.3f} max={rho_colloc.max():.3f}")
print(f"local corr(eta,NOx): min={corr_colloc.min():.3f} max={corr_colloc.max():.3f}")
print(f"validity v_j   : min={validity_eff_nox.min():.3f} max={validity_eff_nox.max():.3f} "
      f"(fraction fully active, v=1: {(validity_eff_nox>=0.999).mean():.1%})")


**Visualized on the first two input dimensions** (SOI vs. λ, holding the LHS sample's own values for the other two):

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["density ratio \u03c1(x)/\u03c1_max", "validity v_j(x) for \u03b7-NOx"])
fig.add_trace(go.Scatter(x=X_colloc[:, SOI_IDX], y=X_colloc[:, LAMBDA_IDX], mode="markers",
                          marker=dict(color=rho_colloc, colorscale="Blues", size=5, showscale=True,
                                      colorbar=dict(x=0.46)), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=X_colloc[:, SOI_IDX], y=X_colloc[:, LAMBDA_IDX], mode="markers",
                          marker=dict(color=validity_eff_nox, colorscale="Oranges", size=5, showscale=True),
                          showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="SOI (norm)")
fig.update_yaxes(title_text="lambda (norm)", col=1)
fig.update_layout(height=420, width=900, title_text="Adaptive weighting components over collocation points")
fig.show()


## 9. Wiring check — all six constraints on an untrained network

Not a meaningful physics result yet (random weights), just confirms
every constraint runs end-to-end against a real `keras` model instead
of only the synthetic test harness, using the architecture **B1**
selected.

In [ ]:
import json as _json
selection_path = OUT_DIR / "B1_selected_architecture.json"
if selection_path.exists():
    with open(selection_path) as f:
        hidden_units = tuple(_json.load(f)["hidden_units"])
else:
    hidden_units = ()
    print("WARNING: B1 selection not found, using linear fallback for this wiring check.")

tf.random.set_seed(SEED)
probe_model = keras.Sequential([keras.Input(shape=(N_IN,))])
for units in hidden_units:
    probe_model.add(layers.Dense(units, activation="tanh"))
probe_model.add(layers.Dense(N_OUT, activation="linear"))

predict_fn = lambda x: probe_model(x, training=False)

results = {
    "NOx-SOI (3.9)": float(nox_soi_constraint(predict_fn, X_colloc)),
    "PM-lambda (3.10)": float(pm_lambda_constraint(predict_fn, X_colloc)),
    "HC-lambda (3.11)": float(hc_lambda_constraint(predict_fn, X_colloc)),
    "NOx-PM (3.12)": float(nox_pm_tradeoff_constraint(predict_fn, X_colloc)),
    "eta-NOx (3.13)": float(eff_nox_tradeoff_constraint(predict_fn, X_colloc)),
    "non-negativity (3.14)": float(nonneg_constraint(predict_fn, X_colloc)),
}
wiring_df = pl.DataFrame({"constraint": list(results.keys()), "penalty": list(results.values())})
wiring_df


**As a picture** — no expected pattern yet (random weights), just confirms every value is finite and non-negative:

In [ ]:
fig = go.Figure(go.Bar(x=wiring_df["constraint"], y=wiring_df["penalty"], marker_color="#534AB7"))
fig.update_layout(title="Physics penalties on an untrained network (sanity check only)",
                   yaxis_title="penalty value", width=750, height=420)
fig.show()


## Optional — persist outputs

Saves the collocation points and the two adaptive-weighting arrays so
C2 doesn't have to regenerate the LHS sample (reproducibility: same
`SEED`, but no reason to redo the sampling) or recompute the KDE/local
correlation.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
colloc_df = pl.DataFrame({c: X_colloc[:, i] for i, c in enumerate(INPUT_COLS)})
colloc_df = colloc_df.with_columns([
    pl.Series("density_ratio", rho_colloc),
    pl.Series("validity_eff_nox", validity_eff_nox),
])
colloc_df.write_csv(OUT_DIR / "C1_collocation_points.csv")
print(f"Saved to {OUT_DIR}")


## Next

**C2** assembles all six penalties plus the data loss and
regularization into the composite loss (Eq. 3.16), sets the base
weights $\lambda_j^0$, and calibrates them so no single term dominates
at initialization — this notebook only builds and tests the pieces,
C2 is where they get combined and weighted.
